# Deploy DICOMweb Gateway App

Full DICOMweb server (QIDO-RS, WADO-RS, WADO-URI, STOW-RS)
backed by Databricks SQL, Lakebase, and Volumes.

In [0]:
%run ./config/proxy_prep

In [0]:
sql_warehouse_id, table, volume = init_widgets(show_volume=True)
init_env()

In [ ]:
app_gateway_name = "pixels-dicomweb-gateway"
lakebase_instance_name = dbutils.widgets.get("lakebase_instance_name")

dbutils.widgets.text(
    "nifti_segmentation_table",
    "",
    label="7.0 (optional) UC Delta table for NIfTI segmentation overlays — leave empty to disable /nifti routes",
)
nifti_segmentation_table = dbutils.widgets.get("nifti_segmentation_table")

# Generate app.yml and Refresh Resources

In [0]:
import os

# Compute repo root from notebook path
_nb_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_nb_dir = "/Workspace" + os.path.dirname(_nb_ctx.notebookPath().get())
_repo_root = os.path.normpath(os.path.join(_nb_dir, ".."))

_dicomweb_gateway_path = os.path.join(_repo_root, "apps", "dicom-web-gateway")

# Derive volume paths
_vol_parts = volume.split(".")
_stow_volume_path = f"/Volumes/{_vol_parts[0]}/{_vol_parts[1]}/{_vol_parts[2]}/stow/"

# Generate app.yml from template
with open(f"{_dicomweb_gateway_path}/app-config.yml", "r") as config_input:
    with open(f"{_dicomweb_gateway_path}/app.yml", "w") as config_output:
        config_output.write(
            config_input.read()
            .replace("{PIXELS_TABLE}", table)
            .replace("{LAKEBASE_INSTANCE_NAME}", lakebase_instance_name)
            .replace("{STOW_VOLUME_PATH}", _stow_volume_path)
            .replace("{NIFTI_SEGMENTATION_TABLE}", nifti_segmentation_table)
        )
print(f"Generated app.yml for gateway")
if nifti_segmentation_table:
    print(f"  NIfTI overlay routes enabled (table={nifti_segmentation_table})")
else:
    print("  NIfTI overlay routes disabled (nifti_segmentation_table widget is empty)")

In [ ]:
from databricks.sdk.service.apps import (
    App,
    AppDeployment,
    AppResource,
    AppResourceSqlWarehouse,
    AppResourceSqlWarehouseSqlWarehousePermission,
)

sql_resource = AppResource(
    name="sql_warehouse",
    sql_warehouse=AppResourceSqlWarehouse(
        id=sql_warehouse_id,
        permission=AppResourceSqlWarehouseSqlWarehousePermission.CAN_USE,
    ),
)

# Gateway makes OBO calls to SQL warehouses and Volumes; the proxy-injected
# x-forwarded-access-token must carry these scopes for dbsql.connect() and
# Volume reads/writes to succeed when the request originates from the viewer
# app (cross-app OBO) or directly from a user.
GATEWAY_USER_API_SCOPES = ["sql", "files.files"]

# The app shell + SP are minted by create-apps.ipynb upstream; here we just
# refresh resources and user_api_scopes before deploying code.
app = w.apps.update(
    app_gateway_name,
    App(
        app_gateway_name,
        resources=[sql_resource],
        user_api_scopes=GATEWAY_USER_API_SCOPES,
    ),
)
print(f"✓ Refreshed resources for app '{app_gateway_name}'")

# Deploy

In [0]:
app_gateway_deploy = w.apps.deploy_and_wait(app_gateway_name, AppDeployment(source_code_path=_dicomweb_gateway_path))
print(f"✅ Gateway deployed: {app_gateway_deploy.status.message}")
print(f"   URL: {w.apps.get(app_gateway_name).url}")